# Imports 

In [ ]:
import os
import pandas as pd

# Functions

# Datasets

In [30]:
path = "../../Data_Processing/outputs/all_banks"

In [31]:
df_new = pd.concat(
    [
        pd.read_pickle(os.path.join(path, file))
        for file in os.listdir(path)
        if "df_banks_part_" in file
    ],
    axis=0,
).reset_index(drop=True)

In [32]:
df_cols = pd.Series(df_new.columns)

# Adjust columns

## Metadata

df["year"] = df["quarter"].str[:4].astype(int)
df["month"] = df["quarter"].str[4:6].astype(int)
df["nr_quarter"] = ((df["month"] - 1) // 3) + 1

## Cummulative to per quarter features

Based on the reviewed documentation, we identified that the reported income statement figures (e.g., revenue, costs, profit) represent year-to-date values for each fiscal year. This means that each quarter’s figures are cumulative for that year, resetting at the beginning of the first quarter.

In contrast, the balance sheet figures (e.g., assets, liabilities, equity) reflect the position at the end of each quarter and do not reset at the start of the year.

Therefore, for income statement data, we chose to estimate the quarterly contribution to eliminate the seasonality effect caused by the reset in the first quarter of each year.


Sources:
- FR Y -9c 
    - financial statement: https://www.federalreserve.gov/reportforms/forms/FR_Y-9C20180930_f.pdf
    - detail on financial statement rubrics: https://www.federalreserve.gov/apps/reportingforms/Download/DownloadAttachment?guid=d036ea09-75d3-4f2f-8fe3-0f43c76d2f70

- FR_Y-9LP (Parent Company Only Financial Statements for Large Bank Holding Companies): 
    - financial statement: https://www.federalreserve.gov/apps/reportingforms/Download/DownloadAttachment?guid=e56cd829-8369-4263-96ca-4de4e12ce585#:~:text=03/2024-,For%20Federal%20Reserve%20Bank%20Use%20Only,for%20investments%20in%20equity%20securities.
    - detail on financial statement rubrics: https://www.federalreserve.gov/reportforms/forms/FR_Y-9LP20220705_i.pdf 

- FR Y-9SP (Parent Company Only Financial Statements for Small Holding Companies) - only reports data semi-annually 
    - financial statement: https://www.federalreserve.gov/apps/reportingforms/Download/DownloadAttachment?guid=9df5bb23-468c-4407-a874-7beaba03b930
    - detail on financial statement rubrics: https://www.federalreserve.gov/apps/reportingforms/Download/DownloadAttachment?guid=05505b4f-780d-49e0-a123-59e7cbf29afb


df_cols = pd.Series(df.columns)


list_cols_cummulative = df_cols[
    df_cols.str.startswith(
        ("REVENUE", "NET_INCOME", "CHARGED", "RECOVERIES", "LOANS", "REVENUE", "EBT")
    )
].unique()

def calc_per_quarter_value(list_report_col, df):
    """
    Convert cumulative reporting columns into per-quarter values while
    handling late-starting companies (first reported quarter > baseline).
    """

    df = df.sort_values(["id", "year", "nr_quarter"]).copy()

    for report_col in list_report_col:

        # determine baseline
        if "FRY9SP" in report_col:
            quarter_baseline = 2
        elif "FRY9LP" in report_col or "FRY9C" in report_col:
            quarter_baseline = 1
        else:
            quarter_baseline = 1

        cum_col = f"{report_col}_CUMMULATIVE"
        df[cum_col] = df[report_col]

        # main diff per id + year
        diff_vals = df.groupby(["id", "year"])[report_col].diff()
        df[report_col] = diff_vals

        # rule 1: if nr_quarter <= baseline → keep cumulative
        baseline_mask = df["nr_quarter"] <= quarter_baseline
        df.loc[baseline_mask, report_col] = df.loc[baseline_mask, cum_col]

        # rule 2: if company first reports AFTER baseline → keep cumulative at that first report
        # first observed quarter within (id, year)
        first_quarter = df.groupby(["id", "year"])["nr_quarter"].transform("min")
        first_obs_mask = (df["nr_quarter"] == first_quarter) & (
            df["nr_quarter"] > quarter_baseline
        )
        df.loc[first_obs_mask, report_col] = df.loc[first_obs_mask, cum_col]

    return df

df_new = calc_per_quarter_value(list_report_col=list_cols_cummulative, df=df)
df_new.head()

## Consolidate per report columns into a single one

In [33]:
df_cols_all = pd.DataFrame(
    (df_cols.str.split("_").str[:-1].apply("_".join), df_cols.str.split("_").str[-1])
).T

### Define FRY9C report data

In [34]:
df_9c_aux = df_new.loc[
    :, [col for col in df_new.columns if col.upper() != col or col.endswith("_FRY9C")]
]

In [35]:
# Identify columns that start with TOTAL_ or REVENUE or NET_INCOME
cols_to_check = [
    col
    for col in df_9c_aux.columns
    if col.startswith("TOTAL_")
    or col.startswith("REVENUE")
    or col.startswith("NET_INCOME")
]

# Group by 'id' and check if all values in these columns are zero for each group
# group_check = df_9c_aux.groupby(["id","quarter"])[cols_to_check].apply(
#     lambda x: (x == 0).all().all()
# )
group_check = (
    df_9c_aux.groupby(["id", "quarter"])[cols_to_check].sum() == 0
).sum(axis=1) == len(cols_to_check)

# IDs that always have zeros across all rows
ids_never_9c = group_check[group_check].index.tolist()

# IDs that have at least one non-zero value
ids_at_least_once9c = group_check[~group_check].index.tolist()

In [36]:
# df_9c = df_new[df_new["id"].isin(ids_at_least_once9c)].loc[
#     :, [col for col in df_new.columns if col.upper() != col or col.endswith("_FRY9C")]
# ]
df_9c = df_new[
    df_new[["id", "quarter"]].apply(tuple, axis=1).isin(ids_at_least_once9c)
].loc[
    :, [col for col in df_new.columns if col.upper() != col or col.endswith("_FRY9C") or col.endswith("_FRY9C_CUMMULATIVE")]
]

df_9c.columns = [
    col.replace('_FRY9C','') 
    for col in df_9c.columns]
df_9c["report_type"] = "FRY9C"

### Define parent company SP data

In [37]:
df_sp_aux = df_new.loc[
    :,
    [
        col
        for col in df_new.columns
        if col.upper() != col
        or col.endswith("_FRY9SP")
        or col.endswith("_FRY9SP_CUMMULATIVE")
    ],
]

In [38]:
cols_to_check_sp = [
    col
    for col in df_sp_aux.columns
    if (
        col.startswith("TOTAL_")
        or col.startswith("REVENUE")
        or col.startswith("NET_INCOME")
    )
    and ((col.endswith("_FRY9SP_CUMMULATIVE") or col.endswith("_FRY9SP")))
]

In [39]:
# # df_sp_aux = df_new.loc[:,cols_to_check_sp+['id','quarter']].groupby(['id','quarter']).sum().sum(axis=1)
# df_sp_aux = (
#     df_new.loc[:, cols_to_check_sp + ["id", "quarter"]]
#     .groupby(["id", "quarter"])
#     .sum()
#     .stack()
# )
# idxs_sp = (
#     df_sp_aux[df_sp_aux != 0]
#     .reset_index()
#     .loc[:, ["id", "quarter"]]
#     .apply(tuple, axis=1)
#     .tolist()
# )

# df_sp_aux = df_new.loc[:,cols_to_check_sp+['id','quarter']].groupby(['id','quarter']).sum().sum(axis=1)
group_check_sp = (
    df_sp_aux.groupby(["id", "quarter"])[cols_to_check_sp].sum() == 0
).sum(axis=1) == len(cols_to_check_sp)

# IDs that always have zeros across all rows
ids_never_sp = group_check_sp[group_check_sp].index.tolist()

# IDs that have at least one non-zero value
ids_sp = group_check_sp[~group_check_sp].index.tolist()

In [40]:
# # Define a function to assign report_type
# def get_report_type(row,ids_sp, ids_lp):
#     pair = (row["id"], row["quarter"])
#     if pair in ids_lp and pair in ids_sp:
#         return "FRY9SP_FRY9LP"
#     elif pair in ids_sp:
#         return "FRY9SP"
#     elif pair in ids_lp:
#         return "FRY9LP"
#     else:
#         return "FRY9SP_FRY9LP"  # Default if not in either set


df_sp = (
    df_new[df_new[["id", "quarter"]].apply(tuple, axis=1).isin(ids_sp)]
    .loc[
        :,
        [
            col
            for col in df_new.columns
            if col.upper() != col or col.endswith("_FRY9SP") or col.endswith("_FRY9SP_CUMMULATIVE")
        ],
    ]
    .copy()
)

# Apply the function to create the new column
df_sp["report_type"] = "FRY9SP"
df_sp.columns = [
    col.replace('_FRY9SP','')
    # col.split("_FRY9SP")[0]
    for col in df_sp.columns
    # if col.upper() != col or col.endswith("_FRY9SP")
]

### Define parent company LP data

In [41]:
df_lp_aux = df_new.loc[
    :,
    [
        col
        for col in df_new.columns
        if col.upper() != col
        or col.endswith("_FRY9LP")
        or col.endswith("_FRY9LP_CUMMULATIVE")
    ],
]

In [42]:
cols_to_check_lp = [
    col
    for col in df_lp_aux.columns
    if (
        col.startswith("TOTAL_")
        or col.startswith("REVENUE")
        or col.startswith("NET_INCOME")
    )
    and ((col.endswith("_FRY9LP_CUMMULATIVE") or col.endswith("_FRY9LP")))
]

In [43]:
group_check_lp = (
    df_lp_aux.groupby(["id", "quarter"])[cols_to_check_lp].sum() == 0
).sum(axis=1) == len(cols_to_check_lp)

# IDs that always have zeros across all rows
ids_never_lp = group_check_lp[group_check_lp].index.tolist()

# IDs that have at least one non-zero value
ids_lp = group_check_lp[~group_check_lp].index.tolist()

In [44]:

df_lp = (
    df_new[df_new[["id", "quarter"]].apply(tuple, axis=1).isin(ids_lp)]
    .loc[
        :,
        [
            col
            for col in df_new.columns
            if col.upper() != col or col.endswith("_FRY9LP") or col.endswith("_FRY9LP_CUMMULATIVE")
        ],
    ]
    .copy()
)

# Apply the function to create the new column
df_lp["report_type"] = "FRY9LP"
df_lp.columns = [
    col.replace('_FRY9LP','')
    # col.split("_FRY9LP")[0]
    for col in df_lp.columns
    # if col.upper() != col or col.endswith("_FRY9LP")
]

### Define parent company (either  SP or LP)

In [45]:
cond_lp_ = df_new[["id", "quarter"]].apply(tuple, axis=1).isin(ids_lp)
cond_sp_ = df_new[["id", "quarter"]].apply(tuple, axis=1).isin(ids_sp)
cond_9c_ = df_new[["id", "quarter"]].apply(tuple, axis=1).isin(ids_at_least_once9c)

df_splp_aux = df_new[
    (cond_lp_ == False) & (cond_sp_ == False) & (cond_9c_ == False)
].copy()

In [46]:
df_splp_aux.loc[:, [col for col in df_splp_aux.columns if col.upper() == col]].fillna(
    0
).eq(0).all(axis=1).sort_values().all()

np.True_

In [47]:
###TODO: ADD splp (banks reporting always zero so we cannot distinguish between sp or lp) to parent company dataset. since they are very few we disregard them for now

### Define parent company dataset

In [48]:
df_parent_company = pd.concat([df_sp, df_lp],axis=0).reset_index(drop=True)

### Save locally parent company and 9c datasets

In [49]:
main_dir = "../outputs/banks_y9c/"

if os.path.exists(main_dir) == False:
    os.makedirs(main_dir, exist_ok=False)

chunk_size = 50_000
for i in range(0, len(df_9c), chunk_size):
    chunk = df_9c.iloc[i : i + chunk_size]
    chunk.to_pickle(f"{main_dir}df_y9c_part_{i//chunk_size + 1}.pickle")

In [50]:
main_dir = "../outputs/banks_parent/"

if os.path.exists(main_dir) == False:
    os.makedirs(main_dir, exist_ok=False)

chunk_size = 50_000
for i in range(0, len(df_parent_company), chunk_size):
    chunk = df_parent_company.iloc[i : i + chunk_size]
    chunk.to_pickle(f"{main_dir}df_parent_part_{i//chunk_size + 1}.pickle")